In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install trl unsloth

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
#import unsloth before torch
import datetime
import torch
from datasets import load_dataset
from huggingface_hub import HfApi, login
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
# Authentication and repo configurationfrom datasets import load_dataset
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")


login(hf_token)

lora_repo = "willy-wonka5745/prog-companion-qwen-3b-lora"
gguf_repo = "willy-wonka5745/prog-companion-qwen-3b-gguf"
max_seq_length = 2048

api = HfApi()
adapter_exists = api.repo_exists(repo_id=lora_repo, token=hf_token)

In [ ]:
# Model loading logic: Day 1 creates base + LoRA, Day 2+ resumes existing LoRA
if adapter_exists:
    print("Found existing checkpoint on Hugging Face. Resuming from LoRA adapter...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=lora_repo,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
        device_map="auto",
    )
    FastLanguageModel.for_training(model)
else:
    print("Day 1 detected. Initializing fresh base model and LoRA adapter...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-Coder-3B-bnb-4bit",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
        device_map="auto",

    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

In [ ]:
"""
# Daily data chunking based on day of year
def format_cp_prompt(examples):
    texts = []
    for desc, sol in zip(examples["description"], examples["solutions"]):
        valid_code = sol["solution"][0] if sol["solution"] else ""
        texts.append(f"Problem:\n{desc}\n\nSolution:\n{valid_code}")
    return {"text": texts}

day_of_year = datetime.datetime.now().timetuple().tm_yday
chunk_size = 5000
start_index = (day_of_year % 100) * chunk_size
end_index = start_index + chunk_size

print(f"Loading dataset slice from {start_index} to {end_index}...")

dataset = load_dataset("deepmind/code_contests", split=f"train[{start_index}:{end_index}]")
dataset = dataset.map(format_cp_prompt, batched=True, num_proc=2)
"""

def format_cp_prompt(examples):
    texts = []
    for desc, sol in zip(examples["description"], examples["solutions"]):
        valid_code = sol["solution"][0] if sol["solution"] else ""
        texts.append(f"Problem:\n{desc}\n\nSolution:\n{valid_code}")
    return {"text": texts}

# Load the full train dataset to get its total length
full_dataset = load_dataset("deepmind/code_contests", split="train")
total_size = len(full_dataset)

# --- MANUAL CHUNK CONTROL ---
# Change this number manually before each run (e.g., 0, 1, 2, 3...)
chunk_index = 2 #chunk 3 on 10 oct and chunk 4 on 11 oct
chunk_size = 3000

# Calculate current slice boundaries
start_index = (chunk_index * chunk_size) % total_size
end_index = min(start_index + chunk_size, total_size)

print(f"Loading dataset slice from {start_index} to {end_index} (Manual Chunk index: {chunk_index})...")

# Select slice and map formatting
dataset = full_dataset.select(range(start_index, end_index))
dataset = dataset.map(format_cp_prompt, batched=True, num_proc=2)

In [ ]:
# Trainer setup for Kaggle limits
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=750,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="qwen_temp",
        save_strategy="no",
    ),
)

In [ ]:
# Run training
print("Starting training run...")
trainer.train()

In [ ]:
# Sync updated weights to Hugging Face
print("Pushing updated LoRA adapter for tomorrow's run...")
model.push_to_hub(lora_repo, token=hf_token)

print("Exporting updated 4-bit GGUF for local inference...")
model.push_to_hub_gguf(
    gguf_repo,
    tokenizer,
    quantization_method="q4_k_m",
    token=hf_token
)

print("Run finished successfully.")
